In [1]:
!pip install datasets ipywidgets peft

In [2]:
pip install transformers accelerate bitsandbytes>0.37.0

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import math
import numpy as np
import torch
from tqdm.auto import tqdm 
from datetime import timedelta
import time
import gc
from peft import prepare_model_for_kbit_training
from torch.nn.utils import prune
import transformers

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    TrainerCallback,
    BitsAndBytesConfig
)
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training, PeftModel

transformers.logging.set_verbosity_info()
torch.backends.cudnn.benchmark = True
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()
gc.collect()

# 1. Load your text dataset from the Kaggle input path
with open('/kaggle/input/book-batch1/cleaned_book_batch1.md', 'r', encoding='utf-8') as f:
    all_lines = f.readlines()

# 2. Load the tokenizer and determine the model's maximum context length
model_name = "Qwen/Qwen2.5-Math-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
max_context_length = tokenizer.model_max_length
print(f"Maximum context length: {max_context_length}")

# Use smaller chunks to further reduce memory pressure
chunk_size = 2048  # Further reduced from 4096

validation_size = 800  # You can adjust this between 500-1000
train_lines = all_lines[:-validation_size]
validation_lines = all_lines[-validation_size:]

print(f"Split dataset into {len(train_lines)} training lines and {len(validation_lines)} validation lines")

# 3. Create corpus strings
train_corpus = "".join(train_lines)
validation_corpus = "".join(validation_lines)

# 3. Tokenize the entire corpus and split it into reasonably sized chunks
# def prepare_corpus_for_training(corpus, tokenizer, chunk_size):
#     tokens = tokenizer(corpus, truncation=False, return_tensors="np")["input_ids"][0]
    
#     total_chunks = math.ceil(len(tokens) / chunk_size)
#     print(f"Total tokens: {len(tokens)}, Creating {total_chunks} chunks of size {chunk_size}")
    
#     chunks = []
#     for i in range(0, len(tokens), chunk_size):
#         chunk = tokens[i:i + chunk_size].tolist()
#         if len(chunk) < chunk_size:
#             chunk = chunk + [tokenizer.pad_token_id] * (chunk_size - len(chunk))
#         chunks.append({"input_ids": chunk})
    
#     return Dataset.from_list(chunks)

def prepare_corpus_for_training(corpus, tokenizer, chunk_size, max_samples=5000):
    # If corpus is a string, take first n lines
    if isinstance(corpus, str):
        corpus = "\n".join(corpus.split("\n")[:max_samples])
    
    tokens = tokenizer(corpus, truncation=False, return_tensors="np")["input_ids"][0]
    
    total_chunks = math.ceil(len(tokens) / chunk_size)
    print(f"Total tokens: {len(tokens)}, Creating {total_chunks} chunks of size {chunk_size}")
    
    chunks = []
    for i in range(0, len(tokens), chunk_size):
        chunk = tokens[i:i + chunk_size].tolist()
        if len(chunk) < chunk_size:
            chunk = chunk + [tokenizer.pad_token_id] * (chunk_size - len(chunk))
        chunks.append({"input_ids": chunk})
    
    return Dataset.from_list(chunks)


train_dataset = prepare_corpus_for_training(train_corpus, tokenizer, chunk_size)
validation_dataset = prepare_corpus_for_training(validation_corpus, tokenizer, chunk_size)


def apply_layer_pruning(model, pruning_ratio=0.3):
    """Apply structured pruning to even-numbered transformer layers"""
    for name, module in model.named_modules():
        if "transformer.h." in name and isinstance(module, torch.nn.Linear):
            layer_num = int(name.split(".")[2])
            if layer_num % 2 == 0:  # Only prune even-numbered layers
                prune.l1_unstructured(module, name='weight', amount=pruning_ratio)
    return model
    
class MinimalProgressCallback(TrainerCallback):
    def __init__(self):
        self.training_start = time.time()
        
    def on_epoch_begin(self, args, state, control, **kwargs):
        print(f"\n{'='*40}\nBeginning Epoch {state.epoch + 1}/{args.num_train_epochs}\n{'='*40}")
        self.epoch_start = time.time()
        
    def on_epoch_end(self, args, state, control, **kwargs):
        epoch_time = time.time() - self.epoch_start
        total_time = time.time() - self.training_start
        remaining = epoch_time * (args.num_train_epochs - state.epoch - 1)
        
        print(f"\n{'='*40}\nEpoch {state.epoch + 1}/{args.num_train_epochs} completed")
        print(f"Time: {timedelta(seconds=int(epoch_time))}, Remaining: {timedelta(seconds=int(remaining))}")
        print(f"GPU Memory: {torch.cuda.memory_allocated() / (1024**3):.1f} GB\n{'='*40}")

    
# Create data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# 5. LoRA-optimized training arguments for continued pretraining
training_args = TrainingArguments(
    output_dir="./qwen_math_nbody_lora",
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=32,
    # Standard learning rate for non-embedding layers
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    dataloader_drop_last=True,
    fp16=True,
    logging_first_step=True,
    gradient_checkpointing=True,
    logging_dir="./logs",
    warmup_ratio=0.2,
    logging_strategy="steps",
    log_level="info",
    report_to=["tensorboard"],
    gradient_checkpointing_kwargs={"use_reentrant": False},
    # Use adamw_hf which better supports parameter groups for decoupled learning rates
    optim="adamw_8bit",
    max_grad_norm=0.5,
    lr_scheduler_type="cosine_with_restarts",
    dataloader_num_workers=4,
    evaluation_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

# 6. Load model for examination to identify all module names
print("Loading model to identify layer structure...")
temp_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    use_cache=False,
    torch_dtype=torch.float16,
    device_map={"": 0}  # Load on first GPU only temporarily
)

# Identify embedding and other linear layers for targeting
embedding_layers = []
linear_layers = []
for name, module in temp_model.named_modules():
    if 'embed' in name.lower() or 'lm_head' in name.lower():
        embedding_layers.append(name)
    elif isinstance(module, torch.nn.Linear) and 'embed' not in name.lower() and 'lm_head' not in name.lower():
        linear_layers.append(name)

print(f"Found {len(embedding_layers)} embedding layers: {embedding_layers}")
print(f"Found {len(linear_layers)} linear layers")

# Clean up memory
del temp_model
torch.cuda.empty_cache()
gc.collect()

# 7. Comprehensive LoRA configuration for continued pretraining
# Combine both standard target_modules and add embedding layers
# List adjusted based on Qwen2.5 specific architecture

book_lora_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

paper_lora_config = LoraConfig(
    r=8, 
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 8. Initialize model with optimized settings
# Load base model first
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Math-1.5B",
    use_cache=False,
    device_map="auto",
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    trust_remote_code=True
)

# model = apply_layer_pruning(model)
model = prepare_model_for_kbit_training(model)

# 9. Apply LoRA to the model
print("Applying LoRA adapters to model...")
model = PeftModel.from_pretrained(
    model, 
    "/kaggle/input/book-adapter/pytorch/default/1", 
    adapter_name="book_adapter"
)
model.set_adapter("book_adapter") 
model.print_trainable_parameters()

# 10. Create custom optimizer with decoupled learning rates
def get_optimizer_grouped_parameters(model, embedding_lr=1e-5, non_embedding_lr=1e-4):
    """Create parameter groups with different learning rates for embeddings vs other layers."""
    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight", "ln_", "norm", "embedding"]
    embedding_names = ["wte", "lm_head"]
    
    optimizer_grouped_parameters = [
        # Embedding params with lower learning rate and no weight decay
        {
            "params": [p for n, p in model.named_parameters() 
                      if any(nd in n for nd in embedding_names) and p.requires_grad],
            "lr": embedding_lr,
            "weight_decay": 0.0,
        },
        # Non-embedding params with regular learning rate and weight decay
        {
            "params": [p for n, p in model.named_parameters() 
                      if not any(nd in n for nd in embedding_names) 
                      and not any(nd in n for nd in no_decay) and p.requires_grad],
            "lr": non_embedding_lr,
            "weight_decay": training_args.weight_decay,
        },
        # Non-embedding params with regular learning rate and no weight decay
        {
            "params": [p for n, p in model.named_parameters() 
                      if not any(nd in n for nd in embedding_names) 
                      and any(nd in n for nd in no_decay) and p.requires_grad],
            "lr": non_embedding_lr,
            "weight_decay": 0.0,
        },
    ]
    return optimizer_grouped_parameters

# 11. Create a custom trainer with decoupled learning rates
class CustomPEFTTrainer(Trainer):
    def create_optimizer(self):
        """Create optimizer with separate learning rates for embedding vs. non-embedding parameters"""
        if self.optimizer is None:
            # Create parameter groups with decoupled learning rates
            embedding_lr = self.args.learning_rate / 10  # Lower embedding LR (10x smaller)
            non_embedding_lr = self.args.learning_rate
            
            print(f"Using decoupled learning rates: embedding={embedding_lr}, other={non_embedding_lr}")
            
            optimizer_grouped_parameters = get_optimizer_grouped_parameters(
                self.model, 
                embedding_lr=embedding_lr,
                non_embedding_lr=non_embedding_lr
            )
            
            # Create optimizer with parameter groups
            self.optimizer = torch.optim.AdamW(
                optimizer_grouped_parameters,
                lr=self.args.learning_rate,
                betas=(0.9, 0.999),
                eps=1e-8,
            )
        
        return self.optimizer

# 12. Initialize custom trainer with all optimizations
progress_callback = MinimalProgressCallback()
trainer = CustomPEFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
)
trainer.add_callback(progress_callback)

tokenizer_config.json:   0%|          | 0.00/7.32k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

loading file vocab.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Math-1.5B/snapshots/4a83ca6e4526a4f2da3aa259ec36c259f66b2ab2/vocab.json
loading file merges.txt from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Math-1.5B/snapshots/4a83ca6e4526a4f2da3aa259ec36c259f66b2ab2/merges.txt
loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Math-1.5B/snapshots/4a83ca6e4526a4f2da3aa259ec36c259f66b2ab2/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Math-1.5B/snapshots/4a83ca6e4526a4f2da3aa259ec36c259f66b2ab2/tokenizer_config.json
loading file chat_template.jinja from cache at None
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Math-1.5B/snapshots/4a83ca6e4526a4f2da3aa259ec36c259f66b2ab2/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "max_new_tokens": 2048
}



Found 2 embedding layers: ['model.embed_tokens', 'lm_head']
Found 196 linear layers


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Math-1.5B/snapshots/4a83ca6e4526a4f2da3aa259ec36c259f66b2ab2/config.json
Model config Qwen2Config {
  "_name_or_path": "Qwen/Qwen2.5-Math-1.5B",
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "max_position_embeddings": 4096,
  "max_window_layers": 21,
  "model_type": "qwen2",
  "num_attention_heads": 12,
  "num_hidden_layers": 28,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 10000,
  "sliding_window": null,
  "tie_word_embeddings": true,
  "torch_dtype": "float16",
  "transformers_version": "4.47.0",
  "use_cache": false,
  "use_mrope": false,
  "use_sliding_window": false,
  "vocab_size": 151936
}

loading weights file model.safetensors from cache at

Applying LoRA adapters to model...
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Using auto half precision backend


In [4]:

# Print training configuration
print(f"\n{'*'*70}")
print(f"ENHANCED LORA CONTINUED PRETRAINING CONFIGURATION:")
print(f"Model: {model_name}")
print(f"LoRA rank: {book_lora_config.r}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Using decoupled learning rates: embedding={training_args.learning_rate/10}, other={training_args.learning_rate}")
print(f"Batch size: {training_args.per_device_train_batch_size} × "
      f"{training_args.gradient_accumulation_steps} steps × "
      f"{torch.cuda.device_count()} GPUs = "
      f"{training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps * torch.cuda.device_count()}")
print(f"{'*'*70}\n")

# Start the training process
print("\n🚀 Starting research-optimized LoRA continued pretraining...\n")
try:
    trainer.train()
    print("\n✅ Training completed successfully!")
    # Save the final model
    print("Saving final model...")
    model.save_pretrained("./qwen_math_nbody_final_lora")
    tokenizer.save_pretrained("./qwen_math_nbody_final_lora")
    print("Model saved at ./qwen_math_nbody_final_lora")
except Exception as e:
    print(f"\n❌ Training interrupted: {e}")
    # Save checkpoint even if interrupted
    print("Saving emergency checkpoint...")
    model.save_pretrained("./qwen_math_nbody_checkpoint_lora")
    tokenizer.save_pretrained("./qwen_math_nbody_checkpoint_lora")
    print("Emergency checkpoint saved at ./qwen_math_nbody_checkpoint_lora")


***** Running training *****
  Num examples = 70
  Num Epochs = 5
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 64
  Gradient Accumulation steps = 32
  Total optimization steps = 5
  Number of trainable parameters = 18,464,768



**********************************************************************
ENHANCED LORA CONTINUED PRETRAINING CONFIGURATION:
Model: Qwen/Qwen2.5-Math-1.5B
LoRA rank: 16
Epochs: 5
Using decoupled learning rates: embedding=2.0000000000000003e-06, other=2e-05
Batch size: 2 × 32 steps × 1 GPUs = 64
**********************************************************************


🚀 Starting research-optimized LoRA continued pretraining...

Using decoupled learning rates: embedding=2.0000000000000003e-06, other=2e-05

Beginning Epoch 1/5


Step,Training Loss,Validation Loss



Epoch 2.0/5 completed
Time: 0:04:56, Remaining: 0:14:48
GPU Memory: 2.2 GB

Beginning Epoch 2.0/5

Epoch 3.0/5 completed
Time: 0:04:55, Remaining: 0:09:50
GPU Memory: 2.2 GB

Beginning Epoch 3.0/5


Saving model checkpoint to ./qwen_math_nbody_lora/checkpoint-5
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Math-1.5B/snapshots/4a83ca6e4526a4f2da3aa259ec36c259f66b2ab2/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "max_position_embeddings": 4096,
  "max_window_layers": 21,
  "model_type": "qwen2",
  "num_attention_heads": 12,
  "num_hidden_layers": 28,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 10000,
  "sliding_window": null,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.47.0",
  "use_cache": true,
  "use_mrope": false,
  "use_sliding_window": false,
  "vocab_size": 151936
}



Training completed. Do not forge


Epoch 3.914285714285714/5 completed
Time: 0:04:31, Remaining: 0:04:54
GPU Memory: 2.2 GB

✅ Training completed successfully!
Saving final model...


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Math-1.5B/snapshots/4a83ca6e4526a4f2da3aa259ec36c259f66b2ab2/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "max_position_embeddings": 4096,
  "max_window_layers": 21,
  "model_type": "qwen2",
  "num_attention_heads": 12,
  "num_hidden_layers": 28,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 10000,
  "sliding_window": null,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.47.0",
  "use_cache": true,
  "use_mrope": false,
  "use_sliding_window": false,
  "vocab_size": 151936
}

tokenizer config file saved in ./qwen_math_nbody_final_lora/tokenizer_config.json
Special tokens 

Model saved at ./qwen_math_nbody_final_lora


In [5]:
log_history = trainer.state.log_history
print("Training and evaluation history:")
print(log_history)

Training and evaluation history:
[{'loss': 36.5786, 'grad_norm': 1.4407353401184082, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.9142857142857143, 'step': 1}, {'train_runtime': 862.9946, 'train_samples_per_second': 0.406, 'train_steps_per_second': 0.006, 'total_flos': 3330985533898752.0, 'train_loss': 22.826182556152343, 'epoch': 2.914285714285714, 'step': 5}]


In [6]:
# import os
# import json
# import shutil
# from peft import PeftModel
# from transformers import AutoModelForCausalLM, AutoTokenizer

# # 1. Load the base model
# base_model_id = "Qwen/Qwen2.5-Math-1.5B"
# model = AutoModelForCausalLM.from_pretrained(
#     base_model_id,
#     torch_dtype="auto",
#     device_map="auto"
# )
# tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# # 2. Create a temporary directory to combine both adapter files
# temp_adapter_dir = "/kaggle/working/combined_adapter"
# os.makedirs(temp_adapter_dir, exist_ok=True)

# # 3. Copy the adapter weights file
# weights_source = "/kaggle/input/central_config_adapter/pytorch/default/1/adapter_model.safetensors"
# weights_dest = os.path.join(temp_adapter_dir, "adapter_model.safetensors")
# shutil.copy(weights_source, weights_dest)

# # 4. Copy or create the config file
# # Assuming you've uploaded the config JSON to a different location
# config_source = "/kaggle/input/adapter-config/adapter_config.json"  # Update this path
# config_dest = os.path.join(temp_adapter_dir, "adapter_config.json")
# shutil.copy(config_source, config_dest)

# # 5. Load the model with the combined adapter
# model = PeftModel.from_pretrained(model, temp_adapter_dir)

# # 6. Test the model
# prompt = "What are four body central configurations? How many four body central configurations exist?"
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# outputs = model.generate(
#     **inputs,
#     max_new_tokens=2048,  # Increased for more detailed explanation
#     temperature=0.6,     # Lower temperature for more precise outputs
#     # do_sample=True,
#     # top_p=0.95
# )
# response = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print(response)

In [7]:
# prompt = '''Explore the unique properties of spiderweb central configurations as defined in celestial mechanics, where masses lie at intersection points of concentric circles with lines meeting at equal angles at the center. Based on the available information:

# 1. Explain how the mass distribution in spiderweb configurations differs from other central configurations
# 2. Analyze Saari's findings about rotation curves and mass distribution approximations in these configurations
# 3. Discuss the potential applications of spiderweb central configurations for understanding galactic structures
# 4. Identify specific open questions about mass distribution in spiderweb configurations that merit further study

# Focus specifically on the spiderweb geometry and avoid generic discussions of n-body problems that don't relate to this specific configuration.'''
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# outputs = model.generate(
#     **inputs,
#     max_new_tokens=2048,  # Increased for more detailed explanation
#     temperature=0.6,     # Lower temperature for more precise outputs
#     # do_sample=True,
#     # top_p=0.95
# )
# response = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print(response)

In [8]:
# prompt = "Explain the concept of bifurcation and the stacking of central configurations in the planar $1+4$ body problem. Include relevant mathematical equations or expressions."
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# outputs = model.generate(
#     **inputs,
#     max_new_tokens=2048,  # Increased for more detailed explanation
#     temperature=0.6,     # Lower temperature for more precise outputs
#     # do_sample=True,
#     # top_p=0.95
# )
# response = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print(response)

In [9]:
# prompt = (
#     "You are an expert on central configurations in mathematical physics. "
#     "Please explain what co-circular central configurations are and show one derivation equation step-by-step."
# )

# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# outputs = model.generate(
#     **inputs,
#     max_new_tokens=2048,  # Increased for more detailed explanation
#     temperature=0.7,     # Lower temperature for more precise outputs
#     do_sample=True,
#     top_p=0.95,
#     repetition_penalty = 1.1
# )
# response = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print(response)

In [10]:
# # Load model directly
# from transformers import AutoTokenizer, AutoModelForCausalLM

# tokenizer = AutoTokenizer.from_pretrained("facebook/galactica-1.3b")
# model = AutoModelForCausalLM.from_pretrained("facebook/galactica-1.3b")

In [11]:
# # Move the model to GPU
# model = model.to("cuda")

# input_text = ("Explain the concept of bifurcation and the stacking of central configurations in the planar "
#               "$1+4$ body problem. Include relevant mathematical equations or expressions.")
# # Transfer input tokens to GPU
# input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to("cuda")

# # Generate output tokens
# outputs = model.generate(input_ids, max_length=1000)

# # Decode and print the generated text
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))